In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/Orginal.csv


# Predicting Irrigation Need
**Kaggle Playground Series S6E4 | April 2026**

## Architecture

- **Snap Features** - cKDTree snap to nearest original value (removes synthetic noise)
- **Inner-fold TE** - 11 stats per cat feature, computed inside each fold (leak-free)
- **20-fold SKF** - Stable CV, no refit
- **XGBoost** (cuda) + **LightGBM** (gpu) + **CatBoost** (GPU) + **YDF** (depth=2) + **cuML RF**
- **RealMLP** via pytabkit, n_ens=8, label smoothing
- **Hill Climbing** - optimal ensemble weights on OOF
- **Pseudo Labels** - TRES=0.999 (very high-confidence only)

## 1. Libraries & Setup

In [2]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'pytabkit', 'ydf'], check=False)

import numpy as np
import pandas as pd
import os, warnings, gc, time
from itertools import combinations
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

from scipy.stats import rankdata
from scipy.spatial import cKDTree

from sklearn.preprocessing import OrdinalEncoder, TargetEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb_lib
from catboost import CatBoostClassifier, Pool

try:
    import ydf
    YDF_AVAIL = True
    print("YDF available")
except ImportError:
    YDF_AVAIL = False
    print("YDF not available - skipping")

try:
    from cuml.ensemble import RandomForestClassifier as cuRF
    import cudf
    CUML_AVAIL = True
    print("cuML available")
except ImportError:
    from sklearn.ensemble import RandomForestClassifier as cuRF
    CUML_AVAIL = False
    print("cuML not available - using sklearn RF fallback")

try:
    from pytabkit import RealMLP_TD_Classifier
    REALMLP_AVAIL = True
    print("RealMLP available")
except ImportError:
    REALMLP_AVAIL = False
    print("RealMLP not available - skipping")

SEED      = 42
N_FOLDS   = 20
TRES      = 0.999
ES_XGB    = 300
ES_LGB    = 200
ES_CAT    = 100
N_CLASSES = 3
ORIG_ROW_WEIGHT = 0.35   # weight original rows less to reduce drift

# Label ordering
INTERNAL_ORDER = ['Low', 'Medium', 'High']   # our internal: 0=Low 1=Med 2=High
PUBLIC_ORDER   = ['High', 'Low', 'Medium']   # kaggle alphabetical

LABEL_MAP = {l: i for i, l in enumerate(INTERNAL_ORDER)}
INV_MAP   = {i: l for i, l in enumerate(INTERNAL_ORDER)}
PUBLIC_MAP = {l: i for i, l in enumerate(PUBLIC_ORDER)}
# internal idx → public idx
I2P = {LABEL_MAP[l]: PUBLIC_MAP[l] for l in INTERNAL_ORDER}

np.random.seed(SEED)
print(f"\nSEED={SEED} | N_FOLDS={N_FOLDS} | TRES={TRES} | ORIG_WEIGHT={ORIG_ROW_WEIGHT}")
print("All libraries loaded!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 8.0 MB/s eta 0:00:00
YDF available
cuML available
RealMLP available

SEED=42 | N_FOLDS=20 | TRES=0.999 | ORIG_WEIGHT=0.35
All libraries loaded!


## 2. Data Loading

In [3]:
COMP      = '/kaggle/input/competitions/playground-series-s6e4'
ORIG_PATH = '/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/Orginal.csv'

train  = pd.read_csv(f'{COMP}/train.csv', index_col='id')
test   = pd.read_csv(f'{COMP}/test.csv',  index_col='id')
sample = pd.read_csv(f'{COMP}/sample_submission.csv')

# Load original dataset
try:
    orig_df = pd.read_csv(ORIG_PATH)
    if 'id' in orig_df.columns:
        orig_df = orig_df.drop(columns=['id'])
    orig_df.index = range(900000, 900000 + len(orig_df))
    ORIG_AVAIL = True
    print(f"Original dataset loaded: {orig_df.shape}")
    print(f"  Class dist: {orig_df['Irrigation_Need'].value_counts(normalize=True).round(3).to_dict()}")
except Exception as e:
    orig_df    = None
    ORIG_AVAIL = False
    print(f"Original dataset not found: {e}")

TARGET   = 'Irrigation_Need'
NUM_COLS = ['Soil_pH','Soil_Moisture','Organic_Carbon','Electrical_Conductivity',
            'Temperature_C','Humidity','Rainfall_mm','Sunlight_Hours',
            'Wind_Speed_kmh','Field_Area_hectare','Previous_Irrigation_mm']
CAT_COLS = ['Soil_Type','Crop_Type','Crop_Growth_Stage','Season',
            'Irrigation_Type','Water_Source','Mulching_Used','Region']
ALL_SOURCE_COLS = NUM_COLS + CAT_COLS

# Combine original + synthetic for training
if ORIG_AVAIL:
    orig_for_train = orig_df[NUM_COLS + CAT_COLS + [TARGET]].copy()
    train_combined = pd.concat([train, orig_for_train], ignore_index=False)
    print(f"Train synthetic: {train.shape} | Train+Original: {train_combined.shape}")
else:
    train_combined = train.copy()
    print("Using synthetic train only")

y_all   = train_combined[TARGET].map(LABEL_MAP).values
N_TRAIN = len(train_combined)
N_TEST  = len(test)
N_ORIG  = len(orig_df) if ORIG_AVAIL else 0
N_SYNTH = N_TRAIN - N_ORIG

print(f"\nFinal Train: {train_combined.shape} | Test: {test.shape}")
print(f"Class: Low={np.mean(y_all==0):.1%}  Medium={np.mean(y_all==1):.1%}  High={np.mean(y_all==2):.1%}")

Original dataset loaded: (10000, 20)
  Class dist: {'Low': 0.586, 'Medium': 0.38, 'High': 0.034}
Train synthetic: (630000, 20) | Train+Original: (640000, 20)

Final Train: (640000, 20) | Test: (270000, 19)
Class: Low=58.7%  Medium=37.9%  High=3.3%


## 3. Snap Features (Drift Correction)

Adversarial CV revealed data drift between synthetic and original datasets:
- `Rainfall_mm`: 0.636 (highest drift)
- `Previous_Irrigation_mm`: 0.582
- `Soil_pH`: 0.573
- `Electrical_Conductivity`: 0.563

**Strategy:** Use `cKDTree` to snap each synthetic value to the nearest value that exists in the original dataset.
This removes synthetic generator noise and re-aligns distributions.

Two new features per column:
- `{col}_snap` - the nearest original value
- `{col}_snap_diff` - synthetic minus snap (the noise signal)

In [4]:
SNAP_COLS = ['Rainfall_mm','Previous_Irrigation_mm','Soil_pH','Electrical_Conductivity']

def add_snap_features(df_train, df_test, orig, snap_cols):
    tr = df_train.copy()
    te = df_test.copy()
    if orig is None:
        print("Skipping snap - original dataset not available")
        return tr, te
    for col in snap_cols:
        if col not in orig.columns:
            continue
        orig_vals = orig[col].dropna().values.reshape(-1, 1)
        tree      = cKDTree(orig_vals)
        _, idx_tr = tree.query(tr[col].values.reshape(-1, 1), k=1)
        _, idx_te = tree.query(te[col].values.reshape(-1, 1), k=1)
        tr[f'{col}_snap']      = orig_vals[idx_tr].flatten()
        tr[f'{col}_snap_diff'] = tr[col] - tr[f'{col}_snap']
        te[f'{col}_snap']      = orig_vals[idx_te].flatten()
        te[f'{col}_snap_diff'] = te[col] - te[f'{col}_snap']
        print(f"  {col}: snap + diff added")
    return tr, te

train_s, test_s = add_snap_features(train_combined, test, orig_df, SNAP_COLS)
SNAP_FEAT_COLS  = [f'{c}_{s}' for c in SNAP_COLS for s in ('snap','snap_diff')]
print(f"Snap features: {SNAP_FEAT_COLS}")

  Rainfall_mm: snap + diff added
  Previous_Irrigation_mm: snap + diff added
  Soil_pH: snap + diff added
  Electrical_Conductivity: snap + diff added
Snap features: ['Rainfall_mm_snap', 'Rainfall_mm_snap_diff', 'Previous_Irrigation_mm_snap', 'Previous_Irrigation_mm_snap_diff', 'Soil_pH_snap', 'Soil_pH_snap_diff', 'Electrical_Conductivity_snap', 'Electrical_Conductivity_snap_diff']


## 4. Feature Engineering

In [5]:
def engineer_features(df):
    d = df.copy()
    # Top-signal interactions (from feature importance analysis)
    d['Moisture_Temp_ratio']   = d['Soil_Moisture'] / (d['Temperature_C'] + 1)
    d['moist_wind']            = d['Soil_Moisture'] / (d['Wind_Speed_kmh'] + 1)
    d['moist_rain']            = d['Soil_Moisture'] / (d['Rainfall_mm'] + 1)
    # Evapotranspiration proxy (matches LB 0.97774 notebook)
    d['ET_proxy']              = (d['Temperature_C'] * d['Wind_Speed_kmh'] * d['Sunlight_Hours']) / (d['Humidity'] + 1)
    d['ET_simple']             = d['Temperature_C'] * (1 - d['Humidity']/100) * d['Sunlight_Hours']
    d['Wind_ET_amp']           = d['Wind_Speed_kmh'] * d['ET_simple'] / 100
    d['heat_stress']           = d['Temperature_C'] * d['Sunlight_Hours']
    d['drying_force']          = d['Wind_Speed_kmh'] * d['Temperature_C'] / (d['Humidity'] + 1)
    # Water supply/deficit
    d['water_supply']          = d['Rainfall_mm'] + d['Previous_Irrigation_mm']
    d['water_deficit']         = d['Soil_Moisture'] - d['water_supply'] * 0.1
    d['Rain_vs_prev']          = d['Rainfall_mm'] / (d['Previous_Irrigation_mm'] + 1)
    d['Prev_per_area']         = d['Previous_Irrigation_mm'] / (d['Field_Area_hectare'] + 0.1)
    # Soil health
    d['soil_quality']          = d['Organic_Carbon'] / (d['Electrical_Conductivity'] + 0.1)
    d['OC_Moisture']           = d['Organic_Carbon'] * d['Soil_Moisture']
    d['pH_EC']                 = d['Soil_pH'] * d['Electrical_Conductivity']
    d['Salinity_stress']       = d['Electrical_Conductivity'] / (d['Soil_Moisture'] + 1)
    # Cross-feature multiplications (top-6 features)
    d['moist_x_wind']          = d['Soil_Moisture'] * d['Wind_Speed_kmh']
    d['moist_x_temp']          = d['Soil_Moisture'] * d['Temperature_C']
    d['wind_x_temp']           = d['Wind_Speed_kmh'] * d['Temperature_C']
    d['moisture_sq']           = d['Soil_Moisture'] ** 2
    d['wind_sq']               = d['Wind_Speed_kmh'] ** 2
    d['temp_sq']               = d['Temperature_C'] ** 2
    # Log transforms
    d['log_Rain']              = np.log1p(d['Rainfall_mm'])
    d['log_Prev']              = np.log1p(d['Previous_Irrigation_mm'])
    d['Rain_Moisture_sum']     = d['Rainfall_mm'] / 100 + d['Soil_Moisture']


    # ── DIGIT FEATURES (from top discussion: helps TE binning significantly) ──
    # Extract individual digits of numerical features as categorical signals
    for c in NUM_COLS:
        mx = d[c].max() if hasattr(d[c], 'max') else 9999
        for k in range(-4, 4):
            col_name = f'{c}_digit{k}'
            d[col_name] = ((d[c].values // (10**k)) % 10).astype(np.float32)
    
    # ── EXACT FORMULA FEATURES (from discussion: achieves bACC=1 on original data) ──
    # Binary flags used in the rule
    d['soil_lt_25']  = (d['Soil_Moisture']  < 25).astype(np.float32)
    d['temp_gt_30']  = (d['Temperature_C']  > 30).astype(np.float32)
    d['rain_lt_300'] = (d['Rainfall_mm']    < 300).astype(np.float32)
    d['wind_gt_10']  = (d['Wind_Speed_kmh'] > 10).astype(np.float32)
    # Logit scores from the exact formula
    is_harvest    = (d['Crop_Growth_Stage'] == 'Harvest').astype(float)
    is_sowing     = (d['Crop_Growth_Stage'] == 'Sowing').astype(float)
    is_flowering  = (d['Crop_Growth_Stage'] == 'Flowering').astype(float)
    is_vegetative = (d['Crop_Growth_Stage'] == 'Vegetative').astype(float)
    mulch_yes     = (d['Mulching_Used'] == 'Yes').astype(float)
    mulch_no      = (d['Mulching_Used'] == 'No').astype(float)
    d['logit_High'] = (-20.9697
        + 10.6947 * d['soil_lt_25']  + 5.8763 * d['temp_gt_30']
        + 10.6958 * d['rain_lt_300'] + 5.7444 * d['wind_gt_10']
        + 5.0569  * is_flowering     - 5.3725 * is_harvest
        - 4.8752  * is_sowing        + 5.1283 * is_vegetative
        + 2.8131  * mulch_no         - 2.8755 * mulch_yes)
    d['logit_Low']  = (16.3173
        - 11.0237 * d['soil_lt_25']  - 5.8559 * d['temp_gt_30']
        - 10.8500 * d['rain_lt_300'] - 5.8284 * d['wind_gt_10']
        - 5.4155  * is_flowering     + 5.5073 * is_harvest
        + 5.2299  * is_sowing        - 5.4617 * is_vegetative
        - 3.0014  * mulch_no         + 2.8613 * mulch_yes)
    d['logit_Med']  = (4.6524
        + 0.3290  * d['soil_lt_25']  - 0.0204 * d['temp_gt_30']
        + 0.1542  * d['rain_lt_300'] + 0.0841 * d['wind_gt_10']
        + 0.3586  * is_flowering     - 0.1348 * is_harvest
        - 0.3547  * is_sowing        + 0.3334 * is_vegetative
        + 0.1883  * mulch_no         + 0.0142 * mulch_yes)
    # Simple score rule (High score - Low score)
    d['High_score'] = (2*d['soil_lt_25'] + 2*d['rain_lt_300']
                       + d['temp_gt_30'] + d['wind_gt_10'])
    d['Low_score']  = (2*is_harvest + 2*is_sowing + mulch_yes)
    d['Score_diff'] = d['High_score'] - d['Low_score']
    # Rule-based label as numeric (0=Low, 1=Med, 2=High)
    score = d['Score_diff'].values
    d['formula_label'] = np.where(score > 3, 2, np.where(score > 0, 1, 0)).astype(np.float32)


    # ── MEDIUM-CLASS BOUNDARY FEATURES (from discussion analysis) ──
    # Model confuses Medium with Low/High near Soil_Moisture~25 boundary
    # These soft features help the model separate Medium class
    d['sm_near_25']       = np.abs(d['Soil_Moisture'] - 25).astype(np.float32)
    d['sm_near_25_sq']    = (d['sm_near_25'] ** 2).astype(np.float32)
    d['sm_band_20_30']    = ((d['Soil_Moisture'] >= 20) & (d['Soil_Moisture'] <= 30)).astype(np.float32)
    d['rain_near_300']    = np.abs(d['Rainfall_mm'] - 300).astype(np.float32)
    d['temp_near_30']     = np.abs(d['Temperature_C'] - 30).astype(np.float32)
    d['wind_near_10']     = np.abs(d['Wind_Speed_kmh'] - 10).astype(np.float32)
    # Soft versions of the exact formula thresholds
    d['soil_soft']        = (1 / (1 + np.exp(d['Soil_Moisture'] - 25))).astype(np.float32)
    d['rain_soft']        = (1 / (1 + np.exp(d['Rainfall_mm'] - 300))).astype(np.float32)
    d['temp_soft']        = (1 / (1 + np.exp(30 - d['Temperature_C']))).astype(np.float32)
    d['wind_soft']        = (1 / (1 + np.exp(10 - d['Wind_Speed_kmh']))).astype(np.float32)
    # How many formula conditions are borderline (within 10% of threshold)?
    d['n_borderline']     = (
        (np.abs(d['Soil_Moisture'] - 25) < 5).astype(int) +
        (np.abs(d['Rainfall_mm'] - 300) < 50).astype(int) +
        (np.abs(d['Temperature_C'] - 30) < 3).astype(int) +
        (np.abs(d['Wind_Speed_kmh'] - 10) < 2).astype(int)
    ).astype(np.float32)

    return d

train_fe = engineer_features(train_s)
test_fe  = engineer_features(test_s)

# ── Rounding trick (from discussion: helps TE by creating better bins) ──────
# Round numerics before they go into TE — boosts LightGBM CV by +0.0002
NUM_MAX = train_fe[NUM_COLS].max()
def apply_rounding(df, num_max):
    d = df.copy()
    for c in NUM_COLS:
        mx = num_max[c]
        if mx < 10:
            d[c] = d[c].round(3)
        elif mx < 100:
            d[c] = d[c].round(2)
        else:
            d[c] = d[c].round(1)

    # ── MEDIUM-CLASS BOUNDARY FEATURES (from discussion analysis) ──
    # Model confuses Medium with Low/High near Soil_Moisture~25 boundary
    # These soft features help the model separate Medium class
    d['sm_near_25']       = np.abs(d['Soil_Moisture'] - 25).astype(np.float32)
    d['sm_near_25_sq']    = (d['sm_near_25'] ** 2).astype(np.float32)
    d['sm_band_20_30']    = ((d['Soil_Moisture'] >= 20) & (d['Soil_Moisture'] <= 30)).astype(np.float32)
    d['rain_near_300']    = np.abs(d['Rainfall_mm'] - 300).astype(np.float32)
    d['temp_near_30']     = np.abs(d['Temperature_C'] - 30).astype(np.float32)
    d['wind_near_10']     = np.abs(d['Wind_Speed_kmh'] - 10).astype(np.float32)
    # Soft versions of the exact formula thresholds
    d['soil_soft']        = (1 / (1 + np.exp(d['Soil_Moisture'] - 25))).astype(np.float32)
    d['rain_soft']        = (1 / (1 + np.exp(d['Rainfall_mm'] - 300))).astype(np.float32)
    d['temp_soft']        = (1 / (1 + np.exp(30 - d['Temperature_C']))).astype(np.float32)
    d['wind_soft']        = (1 / (1 + np.exp(10 - d['Wind_Speed_kmh']))).astype(np.float32)
    # How many formula conditions are borderline (within 10% of threshold)?
    d['n_borderline']     = (
        (np.abs(d['Soil_Moisture'] - 25) < 5).astype(int) +
        (np.abs(d['Rainfall_mm'] - 300) < 50).astype(int) +
        (np.abs(d['Temperature_C'] - 30) < 3).astype(int) +
        (np.abs(d['Wind_Speed_kmh'] - 10) < 2).astype(int)
    ).astype(np.float32)

    return d

train_fe = apply_rounding(train_fe, NUM_MAX)
test_fe  = apply_rounding(test_fe, NUM_MAX)
print("Rounding trick applied to numeric cols for better TE binning")

DIGIT_COLS = [f'{c}_digit{k}' for c in NUM_COLS for k in range(-4,4)]
FORMULA_COLS = ['soil_lt_25','temp_gt_30','rain_lt_300','wind_gt_10',
                'logit_High','logit_Low','logit_Med',
                'High_score','Low_score','Score_diff','formula_label']
ENG_COLS = DIGIT_COLS + ['Moisture_Temp_ratio','moist_wind','moist_rain',
            'ET_proxy','ET_simple','Wind_ET_amp','heat_stress','drying_force',
            'water_supply','water_deficit','Rain_vs_prev','Prev_per_area',
            'soil_quality','OC_Moisture','pH_EC','Salinity_stress',
            'moist_x_wind','moist_x_temp','wind_x_temp',
            'moisture_sq','wind_sq','temp_sq',
            'log_Rain','log_Prev','Rain_Moisture_sum'] + FORMULA_COLS
ALL_NUM  = NUM_COLS + SNAP_FEAT_COLS + ENG_COLS
ALL_FEAT = ALL_NUM + CAT_COLS

print(f"Feature counts:")
print(f"  Base numeric    : {len(NUM_COLS)}")
print(f"  Snap features   : {len(SNAP_FEAT_COLS)}")
print(f"  Engineered      : {len(ENG_COLS)}")
print(f"  Categorical     : {len(CAT_COLS)}")
print(f"  ALL_NUM total   : {len(ALL_NUM)}")

# Ordinal encode cats for CatBoost/YDF/cuML (XGB uses raw via TE per fold)
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train_fe[CAT_COLS] = oe.fit_transform(train_fe[CAT_COLS])
test_fe[CAT_COLS]  = oe.transform(test_fe[CAT_COLS])

Rounding trick applied to numeric cols for better TE binning
Feature counts:
  Base numeric    : 11
  Snap features   : 8
  Engineered      : 124
  Categorical     : 8
  ALL_NUM total   : 143


In [6]:
def bal_acc_eval_metric():
    """Custom XGBoost eval metric: balanced accuracy (the actual competition metric).
    Early stopping on true comp metric instead of mlogloss = better models.
    """
    def _m(y_true, y_pred):
        y_pred_labels = np.argmax(y_pred.reshape(-1, N_CLASSES), axis=1)
        return balanced_accuracy_score(y_true.astype(int), y_pred_labels)
    _m.__name__ = 'bal_ACC'
    return _m

def public_preds_with_bias(proba, bias):
    """Convert probabilities → public class indices with log-space bias adjustment."""
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias, axis=1)

def tune_bias(proba_internal, y_internal):
    """
    Greedy coordinate descent on class log-bias offsets.
    Optimizes balanced accuracy directly on OOF predictions.
    Returns: (best_bias, best_score)
    """
    # Convert internal labels to public ordering for final submission
    y_pub = np.array([I2P[v] for v in y_internal], dtype=np.int64)
    # Reorder proba to public ordering
    src = {l: i for i, l in enumerate(INTERNAL_ORDER)}
    proba_pub = proba_internal[:, [src[l] for l in PUBLIC_ORDER]]

    best_bias  = np.zeros(N_CLASSES, dtype=np.float64)
    best_score = balanced_accuracy_score(y_pub, public_preds_with_bias(proba_pub, best_bias))
    print(f"  Before bias: {best_score:.5f}")

    for step in (1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01):
        improved = True
        while improved:
            improved = False
            for ci in range(N_CLASSES):
                for d in (-1.0, 1.0):
                    c = best_bias.copy()
                    c[ci] += d * step
                    s = balanced_accuracy_score(y_pub, public_preds_with_bias(proba_pub, c))
                    if s > best_score + 1e-8:
                        best_bias, best_score, improved = c, s, True
    print(f"  After  bias: {best_score:.5f} | bias={best_bias}")
    return best_bias, best_score, proba_pub, y_pub

print("Bias tuning helpers defined!")
print("  bal_acc_eval_metric() → custom XGB early stopping on true comp metric")
print("  tune_bias() → greedy coord descent on log-space class offsets")
print("  Expected gain from bias tuning: +0.001 to +0.003")

Bias tuning helpers defined!
  bal_acc_eval_metric() → custom XGB early stopping on true comp metric
  tune_bias() → greedy coord descent on log-space class offsets
  Expected gain from bias tuning: +0.001 to +0.003


## 5. Inner-Fold Target Encoding (Leak-Free)

**11 statistics per categorical feature, computed ONLY on the training fold.**
Validation and test rows use the stats from their fold's training data.

- `mean, std, min, max, median` - basic distribution of target per category
- `q05, q10, q45, q55, q90, q95` - quantile profile

Total TE features: 8 cats x 11 stats = **88 TE features**

In [7]:
TE_STATS = ['mean','std','min','max','median','q05','q10','q45','q55','q90','q95']

def compute_te_fold(X_tr, y_tr, X_va, X_te, cat_cols):
    # Compute target encoding stats on train fold - no leakage
    X_tr = X_tr.copy(); X_va = X_va.copy(); X_te = X_te.copy()
    y_s  = pd.Series(y_tr, index=X_tr.index)
    gm   = y_s.mean()
    te_cols = []
    for col in cat_cols:
        grp = y_s.groupby(X_tr[col])
        mapping = {
            f'{col}_te_mean'  : grp.mean(),
            f'{col}_te_std'   : grp.std().fillna(0),
            f'{col}_te_min'   : grp.min(),
            f'{col}_te_max'   : grp.max(),
            f'{col}_te_median': grp.median(),
            f'{col}_te_q05'   : grp.quantile(0.05),
            f'{col}_te_q10'   : grp.quantile(0.10),
            f'{col}_te_q45'   : grp.quantile(0.45),
            f'{col}_te_q55'   : grp.quantile(0.55),
            f'{col}_te_q90'   : grp.quantile(0.90),
            f'{col}_te_q95'   : grp.quantile(0.95),
        }
        for fname, stat_vals in mapping.items():
            X_tr[fname] = X_tr[col].map(stat_vals).fillna(gm)
            X_va[fname] = X_va[col].map(stat_vals).fillna(gm)
            X_te[fname] = X_te[col].map(stat_vals).fillna(gm)
            te_cols.append(fname)
    return X_tr, X_va, X_te, list(dict.fromkeys(te_cols))

# Final feature set = ALL_NUM + TE (no raw cats needed since TE replaces them)
TE_COLS   = [f'{c}_te_{s}' for c in CAT_COLS for s in TE_STATS]
FINAL_FEAT = ALL_NUM + TE_COLS
print(f"TE features       : {len(TE_COLS)}")
print(f"FINAL model feats : {len(FINAL_FEAT)}")

TE features       : 88
FINAL model feats : 231


## 6. CV Framework

In [8]:
SKF = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

X_base     = train_fe[ALL_FEAT].copy()
X_test_b   = test_fe[ALL_FEAT].copy()
y_arr      = y_all.copy()

oof_store  = {}  # model -> (N_TRAIN, N_CLASSES)
test_store = {}  # model -> (N_TEST,  N_CLASSES)
cv_scores  = {}  # model -> list of fold bACC

print(f"CV setup: {N_FOLDS}-fold Stratified KFold | SEED={SEED}")
print(f"Train size : {N_TRAIN:,} | Test size: {N_TEST:,}")
print(f"Feature dim: {X_base.shape[1]}")

CV setup: 20-fold Stratified KFold | SEED=42
Train size : 640,000 | Test size: 270,000
Feature dim: 151


## 7. XGBoost (CUDA)

In [9]:
print("=" * 65)
print(f"XGBoost | {N_FOLDS}-Fold SKF | device=cuda | bal_acc early stop")
print("=" * 65)

# REVERTED to single-seed SEED=42 (V12 approach: CV 0.97322)
# Multi-seed broke because eval_metric in constructor interferes with early stopping
# ── PUBLIC NOTEBOOK PARAMS (max_depth=3, fixed iters, no early stopping) ──
# These match the 0.97955 CV / 0.98011 LB XGB notebook exactly
xgb_params = dict(
    n_estimators     = 3038,        # fixed — matches best public notebook
    learning_rate    = 0.021,
    max_depth        = 3,           # shallower = less overfit on this dataset
    min_child_weight = 3,
    subsample        = 0.734,
    colsample_bytree = 0.505,
    colsample_bylevel= 0.704,
    colsample_bynode = 0.539,
    reg_alpha        = 5.7e-5,
    reg_lambda       = 7.843,
    gamma            = 0.018,
    objective        = 'multi:softprob',
    num_class        = N_CLASSES,
    device           = 'cuda',
    tree_method      = 'hist',
    random_state     = SEED,
    eval_metric      = 'mlogloss',
    # NO early_stopping_rounds — fixed n_estimators for reproducibility
)

oof_xgb  = np.zeros((N_TRAIN, N_CLASSES))
test_xgb = np.zeros((N_TEST,  N_CLASSES))
fold_xgb = []
t0 = time.time()

for fold, (tri, vai) in enumerate(SKF.split(X_base, y_arr)):
    Xtr_r, Xva_r = X_base.iloc[tri].copy(), X_base.iloc[vai].copy()
    Xte_r        = X_test_b.copy()
    ytr, yva     = y_arr[tri], y_arr[vai]

    Xtr, Xva, Xte, te_c = compute_te_fold(Xtr_r, ytr, Xva_r, Xte_r, CAT_COLS)
    fc = ALL_NUM + te_c

    sw = compute_sample_weight('balanced', ytr)
    orig_mask = Xtr_r.index >= 900000
    sw[orig_mask] *= ORIG_ROW_WEIGHT

    m = XGBClassifier(**xgb_params)
    m.fit(Xtr[fc], ytr, sample_weight=sw)

    oof_p  = m.predict_proba(Xva[fc])
    test_p = m.predict_proba(Xte[fc])
    oof_xgb[vai]  = oof_p
    test_xgb     += test_p / N_FOLDS

    s = balanced_accuracy_score(yva, np.argmax(oof_p, axis=1))
    fold_xgb.append(s)
    print(f"  Fold {fold+1:>2}/{N_FOLDS} bACC={s:.5f}")
    del m; gc.collect()

xgb_cv = np.mean(fold_xgb)
oof_store['XGBoost']  = oof_xgb
test_store['XGBoost'] = test_xgb
cv_scores['XGBoost']  = fold_xgb
print(f"\nXGBoost OOF bACC = {xgb_cv:.5f} +/- {np.std(fold_xgb):.5f}")
print(f"Time: {(time.time()-t0)/60:.1f} min")

XGBoost | 20-Fold SKF | device=cuda | bal_acc early stop
  Fold  1/20 bACC=0.97111
  Fold  2/20 bACC=0.97666
  Fold  3/20 bACC=0.97127
  Fold  4/20 bACC=0.97469
  Fold  5/20 bACC=0.97047
  Fold  6/20 bACC=0.97155
  Fold  7/20 bACC=0.97540
  Fold  8/20 bACC=0.97317
  Fold  9/20 bACC=0.97274
  Fold 10/20 bACC=0.97342
  Fold 11/20 bACC=0.97438
  Fold 12/20 bACC=0.97147
  Fold 13/20 bACC=0.97305
  Fold 14/20 bACC=0.97424
  Fold 15/20 bACC=0.96959
  Fold 16/20 bACC=0.97447
  Fold 17/20 bACC=0.97252
  Fold 18/20 bACC=0.97310
  Fold 19/20 bACC=0.97021
  Fold 20/20 bACC=0.96789

XGBoost OOF bACC = 0.97257 +/- 0.00209
Time: 49.4 min


## 8. LightGBM (GPU)

In [10]:
# ── OrderedTE: vectorized CatBoost-style target encoding ─────────────
# Fast pandas-merge implementation — no Python loops over rows

class OrderedTE:
    def __init__(self, smooth=1.0, n_shuffles=4):
        self.smooth     = smooth
        self.n_shuffles = n_shuffles

    def fit_transform(self, X_df, y_arr, cat_cols, seed=42):
        self.cat_cols_ = cat_cols
        self.classes_  = sorted(np.unique(y_arr))
        self.global_pr_= np.array([(y_arr==c).mean() for c in self.classes_],
                                   dtype=np.float32)
        # Store global stats (sum/count per category) for transform()
        self._stats = {}
        tmp = X_df.copy()
        tmp['__y__'] = y_arr
        for c in cat_cols:
            self._stats[c] = {}
            for k, cls in enumerate(self.classes_):
                g = tmp.groupby(c)['__y__'].apply(lambda s: (s==cls).sum()).reset_index()
                g.columns = [c, 'sm']
                g['cnt'] = tmp.groupby(c).size().values
                self._stats[c][cls] = g

        # Build augmented frames
        rng = np.random.RandomState(seed)
        frames = []
        for _ in range(self.n_shuffles):
            idx = rng.permutation(len(X_df))
            Xs  = X_df.iloc[idx].copy().reset_index(drop=True)
            ys  = y_arr[idx]
            Xs  = self._encode_ordered(Xs, ys, cat_cols)
            Xs['__y__'] = ys
            frames.append(Xs)
        return pd.concat(frames, ignore_index=True)

    def _encode_ordered(self, X_df, y_arr, cat_cols):
        X = X_df.copy().reset_index(drop=True)
        for c in cat_cols:
            col_vals = X[c].values
            for k, cls in enumerate(self.classes_):
                y_bin  = (y_arr == cls).astype(np.float32)
                te_out = np.full(len(X), self.global_pr_[k], dtype=np.float32)
                # Vectorized cumsum per group
                order  = np.argsort(col_vals, kind='stable')
                groups = col_vals[order]
                yg     = y_bin[order]
                # Cumulative sum within each group (lagged by 1 for no-leakage)
                cum_sm = np.zeros(len(X), dtype=np.float32)
                cum_cn = np.zeros(len(X), dtype=np.float32)
                prev_cat, start = None, 0
                for i in range(len(X) + 1):
                    if i == len(X) or groups[i] != prev_cat:
                        if prev_cat is not None:
                            seg = slice(start, i)
                            yg_seg = yg[start:i]
                            cs  = np.concatenate([[0.0], np.cumsum(yg_seg)[:-1]])
                            cn  = np.arange(i - start, dtype=np.float32)
                            cum_sm[order[seg]] = cs
                            cum_cn[order[seg]] = cn
                        prev_cat, start = groups[i] if i < len(X) else None, i
                pr = self.smooth * self.global_pr_[k]
                te_out = (cum_sm + pr) / (cum_cn + self.smooth)
                X[f'{c}_OTE_{k}'] = te_out
        return X

    def transform(self, X_df, cat_cols):
        """Fast vectorized transform using pandas merge."""
        X = X_df.copy()
        for c in cat_cols:
            for k, cls in enumerate(self.classes_):
                agg   = self._stats[c][cls]
                prior = float(self.global_pr_[k])
                X     = X.merge(agg, on=c, how='left')
                X[f'{c}_OTE_{k}'] = (
                    (X['sm'].fillna(0) + self.smooth * prior) /
                    (X['cnt'].fillna(0) + self.smooth)
                ).astype(np.float32)
                X.drop(columns=['sm','cnt'], inplace=True, errors='ignore')
        return X

print("OrderedTE ready (vectorized, fast pandas-merge transform)")


OrderedTE ready (vectorized, fast pandas-merge transform)


In [11]:
print("=" * 65)
print(f"LightGBM + OrderedTE | 5-Fold SKF | device=gpu")
print("5 folds (speed): saves ~4h vs 20-fold")
print("=" * 65)

lgbm_params = dict(
    n_estimators     = 6000,
    learning_rate    = 0.05,
    num_leaves       = 32,
    max_depth        = 4,
    colsample_bytree = 0.6,
    subsample        = 0.7,
    subsample_freq   = 1,
    lambda_l1        = 10.0,
    lambda_l2        = 10.0,
    min_child_samples= 100,  # prevents GPU split-count crash
    min_data_in_leaf = 100,
    objective        = 'multiclass',
    num_class        = N_CLASSES,
    metric           = 'multi_logloss',
    device           = 'cpu',  # GPU has unfixable left_count bug with OTE data
    random_state     = SEED,
    verbose          = -1,
    n_jobs           = -1,
)

# 5-fold for LGB (speed) — consistent with YDF
SKF_5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_lgbm  = np.zeros((N_TRAIN, N_CLASSES))
test_lgbm = np.zeros((N_TEST,  N_CLASSES))
fold_lgbm = []
t0 = time.time()

for fold, (tri, vai) in enumerate(SKF_5.split(X_base, y_arr)):
    Xtr_r = X_base.iloc[tri].copy()
    Xva_r = X_base.iloc[vai].copy()
    Xte_r = X_test_b.copy()
    ytr   = y_arr[tri]
    yva   = y_arr[vai]

    # OrderedTE — 4 shuffles, vectorized
    ote     = OrderedTE(smooth=1.0, n_shuffles=4)
    aug     = ote.fit_transform(Xtr_r, ytr, CAT_COLS, seed=SEED + fold)
    ytr_aug = aug.pop('__y__').values
    Xtr_aug = aug

    Xva_enc = ote.transform(Xva_r, CAT_COLS)
    Xte_enc = ote.transform(Xte_r, CAT_COLS)

    # Sample weights — clipped to avoid extreme ratios
    sw_aug = compute_sample_weight('balanced', ytr_aug)
    sw_aug = np.clip(sw_aug, sw_aug.mean() * 0.2, sw_aug.mean() * 5.0)
    sw_aug = sw_aug / sw_aug.mean()

    ote_cols  = [f'{c}_OTE_{k}' for c in CAT_COLS for k in range(N_CLASSES)]
    num_avail = [c for c in ALL_NUM if c in Xtr_aug.columns]
    ote_avail = [c for c in ote_cols if c in Xtr_aug.columns]
    fc_use    = num_avail + ote_avail

    for col in fc_use:
        if col not in Xva_enc.columns: Xva_enc[col] = 0.0
        if col not in Xte_enc.columns: Xte_enc[col] = 0.0

    m = LGBMClassifier(**lgbm_params)
    m.fit(Xtr_aug[fc_use], ytr_aug, sample_weight=sw_aug,
          eval_set=[(Xva_enc[fc_use], yva)],
          callbacks=[lgb_lib.early_stopping(250, verbose=False),
                     lgb_lib.log_evaluation(-1)])

    oof_lgbm[vai]  = m.predict_proba(Xva_enc[fc_use])
    test_lgbm     += m.predict_proba(Xte_enc[fc_use]) / 5

    s = balanced_accuracy_score(yva, np.argmax(oof_lgbm[vai], axis=1))
    fold_lgbm.append(s)
    print(f"  Fold {fold+1:>2}/5 bACC={s:.5f} best_iter={m.best_iteration_}")
    del m; gc.collect()

lgbm_cv = np.mean(fold_lgbm)
oof_store['LightGBM']  = oof_lgbm
test_store['LightGBM'] = test_lgbm
cv_scores['LightGBM']  = fold_lgbm
print(f"\nLightGBM+OTE OOF bACC = {lgbm_cv:.5f} +/- {np.std(fold_lgbm):.5f}")
print(f"Time: {(time.time()-t0)/60:.1f} min")


LightGBM + OrderedTE | 5-Fold SKF | device=gpu
5 folds (speed): saves ~4h vs 20-fold
  Fold  1/5 bACC=0.97026 best_iter=3267
  Fold  2/5 bACC=0.96953 best_iter=2994
  Fold  3/5 bACC=0.96826 best_iter=2945
  Fold  4/5 bACC=0.96925 best_iter=2997
  Fold  5/5 bACC=0.96501 best_iter=2649

LightGBM+OTE OOF bACC = 0.96846 +/- 0.00184
Time: 238.4 min


## 9. CatBoost (GPU)

In [12]:
print("=" * 65)
print(f"CatBoost | {N_FOLDS}-Fold SKF | task_type=GPU")
print("=" * 65)

oof_cat  = np.zeros((N_TRAIN, N_CLASSES))
test_cat = np.zeros((N_TEST,  N_CLASSES))
fold_cat = []
t0 = time.time()

cat_params = dict(
    iterations     = 5000,
    learning_rate  = 0.05,
    depth          = 8,
    l2_leaf_reg    = 4.0,
    loss_function  = 'MultiClass',
    eval_metric    = 'MultiClass',
    random_seed    = SEED,
    task_type      = 'GPU',
    devices        = '0',
    verbose        = 0,
)

for fold, (tri, vai) in enumerate(SKF.split(X_base, y_arr)):
    Xtr_r, Xva_r = X_base.iloc[tri].copy(), X_base.iloc[vai].copy()
    Xte_r        = X_test_b.copy()
    ytr, yva     = y_arr[tri], y_arr[vai]

    Xtr, Xva, Xte, te_c = compute_te_fold(Xtr_r, ytr, Xva_r, Xte_r, CAT_COLS)
    fc = ALL_NUM + te_c

    # Sample weights: class balance + down-weight original rows
    sw = compute_sample_weight('balanced', ytr)
    orig_mask = Xtr_r.index >= 900000
    sw[orig_mask] *= ORIG_ROW_WEIGHT

    cat_feat_idx = [Xtr[fc].columns.get_loc(c) for c in CAT_COLS if c in Xtr[fc].columns]

    m = CatBoostClassifier(**cat_params, early_stopping_rounds=ES_CAT)
    m.fit(
        Pool(Xtr[fc], ytr, cat_features=cat_feat_idx, weight=sw),
        eval_set=Pool(Xva[fc], yva, cat_features=cat_feat_idx),
        use_best_model=True,
    )

    oof_cat[vai]  = m.predict_proba(Pool(Xva[fc], cat_features=cat_feat_idx))
    test_cat     += m.predict_proba(Pool(Xte[fc], cat_features=cat_feat_idx)) / N_FOLDS

    s = balanced_accuracy_score(yva, np.argmax(oof_cat[vai], axis=1))
    fold_cat.append(s)
    print(f"  Fold {fold+1:>2}/{N_FOLDS} bACC={s:.5f} best_iter={m.best_iteration_}")
    del m; gc.collect()

cat_cv = np.mean(fold_cat)
oof_store['CatBoost']  = oof_cat
test_store['CatBoost'] = test_cat
cv_scores['CatBoost']  = fold_cat
print(f"\nCatBoost OOF bACC = {cat_cv:.5f} +/- {np.std(fold_cat):.5f}")
print(f"Time: {(time.time()-t0)/60:.1f} min")

CatBoost | 20-Fold SKF | task_type=GPU
  Fold  1/20 bACC=0.96635 best_iter=4902
  Fold  2/20 bACC=0.97146 best_iter=4574
  Fold  3/20 bACC=0.96800 best_iter=4990
  Fold  4/20 bACC=0.97056 best_iter=4999
  Fold  5/20 bACC=0.96804 best_iter=4872
  Fold  6/20 bACC=0.96735 best_iter=4700
  Fold  7/20 bACC=0.97061 best_iter=4993
  Fold  8/20 bACC=0.96800 best_iter=4790
  Fold  9/20 bACC=0.96648 best_iter=3685
  Fold 10/20 bACC=0.96926 best_iter=4254
  Fold 11/20 bACC=0.96972 best_iter=4964
  Fold 12/20 bACC=0.96790 best_iter=4138
  Fold 13/20 bACC=0.96693 best_iter=4995
  Fold 14/20 bACC=0.97041 best_iter=4902
  Fold 15/20 bACC=0.96548 best_iter=4707
  Fold 16/20 bACC=0.96879 best_iter=4615
  Fold 17/20 bACC=0.96296 best_iter=4994
  Fold 18/20 bACC=0.96943 best_iter=4778
  Fold 19/20 bACC=0.96456 best_iter=4458
  Fold 20/20 bACC=0.96084 best_iter=3964

CatBoost OOF bACC = 0.96766 +/- 0.00262
Time: 40.1 min


## 10. YDF - Yggdrasil Decision Forests (max_depth=2 for diversity)

In [13]:
print("=" * 65)
print(f"YDF | 5-Fold SKF | max_depth=2 (diversity — capped at 5 folds for speed)")
print("=" * 65)

# YDF takes ~7 min/fold → 20 folds = 140 min (too slow)
# Cap at 5 folds — enough diversity signal at much lower cost (~35 min)
YDF_FOLDS = 5
ydf_skf   = StratifiedKFold(n_splits=YDF_FOLDS, shuffle=True, random_state=SEED)

if not YDF_AVAIL:
    print("YDF not available. Skipping.")
    oof_ydf  = None
    test_ydf = None
    fold_ydf = []
else:
    oof_ydf  = np.zeros((N_TRAIN, N_CLASSES))
    test_ydf = np.zeros((N_TEST,  N_CLASSES))
    fold_ydf = []
    t0 = time.time()

    for fold, (tri, vai) in enumerate(ydf_skf.split(X_base, y_arr)):
        Xtr_r, Xva_r = X_base.iloc[tri].copy(), X_base.iloc[vai].copy()
        Xte_r        = X_test_b.copy()
        ytr, yva     = y_arr[tri], y_arr[vai]

        Xtr, Xva, Xte, te_c = compute_te_fold(Xtr_r, ytr, Xva_r, Xte_r, CAT_COLS)
        fc = ALL_NUM + te_c

        Xtr_ydf = Xtr[fc].copy(); Xtr_ydf['label'] = ytr
        Xva_ydf = Xva[fc].copy(); Xva_ydf['label'] = yva

        learner = ydf.GradientBoostedTreesLearner(
            label='label', task=ydf.Task.CLASSIFICATION,
            max_depth=2, num_trees=500, shrinkage=0.1,
            apply_link_function=False,
        )
        m_ydf = learner.train(Xtr_ydf)

        def predict_ydf_proba(model, df_feat, feat_cols, n_classes):
            df_in = df_feat[feat_cols].copy()
            df_in['label'] = 0
            preds = model.predict(df_in)
            if preds.ndim == 1:
                p = np.zeros((len(preds), n_classes))
                p[:, 1] = preds; p[:, 0] = 1 - preds
                return p
            return preds

        oof_p  = predict_ydf_proba(m_ydf, Xva, fc, N_CLASSES)
        test_p = predict_ydf_proba(m_ydf, Xte, fc, N_CLASSES)

        oof_ydf[vai]  += oof_p
        test_ydf      += test_p / YDF_FOLDS

        s = balanced_accuracy_score(yva, np.argmax(oof_p, axis=1))
        fold_ydf.append(s)
        print(f"  Fold {fold+1:>2}/{YDF_FOLDS} bACC={s:.5f}")
        del m_ydf; gc.collect()

    # Scale OOF: each val row predicted once (unlike test averaged over 5 folds)
    ydf_cv = np.mean(fold_ydf)
    oof_store['YDF']  = oof_ydf
    test_store['YDF'] = test_ydf
    cv_scores['YDF']  = fold_ydf
    print(f"\nYDF OOF bACC = {ydf_cv:.5f} +/- {np.std(fold_ydf):.5f}")
    print(f"Time: {(time.time()-t0)/60:.1f} min")

YDF | 5-Fold SKF | max_depth=2 (diversity — capped at 5 folds for speed)
Feature Soil_pH_snap_diff is a NUMERICAL feature with all values recorded in the data spec set to the same value. The feature will likely not be useful during model training.
Feature Electrical_Conductivity_snap_diff is a NUMERICAL feature with all values recorded in the data spec set to the same value. The feature will likely not be useful during model training.
Feature Soil_pH_digit1 is a NUMERICAL feature with all values recorded in the data spec set to the same value. The feature will likely not be useful during model training.
Feature Soil_pH_digit2 is a NUMERICAL feature with all values recorded in the data spec set to the same value. The feature will likely not be useful during model training.
Feature Soil_pH_digit3 is a NUMERICAL feature with all values recorded in the data spec set to the same value. The feature will likely not be useful during model training.
Feature Soil_Moisture_digit2 is a NUMERICAL f

## 11. cuML Random Forest

In [14]:
print("=" * 65)
print(f"cuML RF | {N_FOLDS}-Fold SKF")
print("=" * 65)

# cuML RF does NOT support sample_weight in fit()
# Use class_weight via n_estimators diversity instead
rf_kw = dict(n_estimators=300, max_depth=12, random_state=SEED)
if CUML_AVAIL:
    rf_kw['n_streams'] = 4
else:
    rf_kw['min_samples_leaf'] = 5
    rf_kw['class_weight']     = 'balanced'
    rf_kw['n_jobs']           = -1

oof_rf  = np.zeros((N_TRAIN, N_CLASSES))
test_rf = np.zeros((N_TEST,  N_CLASSES))
fold_rf = []
t0 = time.time()

for fold, (tri, vai) in enumerate(SKF.split(X_base, y_arr)):
    Xtr_r, Xva_r = X_base.iloc[tri].copy(), X_base.iloc[vai].copy()
    Xte_r        = X_test_b.copy()
    ytr, yva     = y_arr[tri], y_arr[vai]

    Xtr, Xva, Xte, te_c = compute_te_fold(Xtr_r, ytr, Xva_r, Xte_r, CAT_COLS)
    fc = ALL_NUM + te_c

    m = cuRF(**rf_kw)
    if CUML_AVAIL:
        # cuML RF: no sample_weight support — use cudf directly
        Xt_cu = cudf.from_pandas(Xtr[fc].astype(np.float32).reset_index(drop=True))
        Xv_cu = cudf.from_pandas(Xva[fc].astype(np.float32).reset_index(drop=True))
        Xe_cu = cudf.from_pandas(Xte[fc].astype(np.float32).reset_index(drop=True))
        yt_cu = cudf.from_pandas(
            pd.Series(ytr.astype(np.int32)).reset_index(drop=True))
        m.fit(Xt_cu, yt_cu)   # no sample_weight — not supported by cuML RF
        oof_p  = m.predict_proba(Xv_cu).to_numpy()
        test_p = m.predict_proba(Xe_cu).to_numpy()
    else:
        sw = compute_sample_weight('balanced', ytr)
        m.fit(Xtr[fc], ytr, sample_weight=sw)
        oof_p  = m.predict_proba(Xva[fc])
        test_p = m.predict_proba(Xte[fc])

    oof_rf[vai]  = oof_p
    test_rf     += test_p / N_FOLDS

    s = balanced_accuracy_score(yva, np.argmax(oof_p, axis=1))
    fold_rf.append(s)
    print(f"  Fold {fold+1:>2}/{N_FOLDS} bACC={s:.5f}")
    del m; gc.collect()

rf_cv = np.mean(fold_rf)
oof_store['cuML_RF']  = oof_rf
test_store['cuML_RF'] = test_rf
cv_scores['cuML_RF']  = fold_rf
print(f"\ncuML RF OOF bACC = {rf_cv:.5f} +/- {np.std(fold_rf):.5f}")
print(f"Time: {(time.time()-t0)/60:.1f} min")

cuML RF | 20-Fold SKF
  Fold  1/20 bACC=0.95934
  Fold  2/20 bACC=0.96664
  Fold  3/20 bACC=0.96250
  Fold  4/20 bACC=0.96473
  Fold  5/20 bACC=0.96402
  Fold  6/20 bACC=0.96378
  Fold  7/20 bACC=0.96378
  Fold  8/20 bACC=0.96151
  Fold  9/20 bACC=0.95632
  Fold 10/20 bACC=0.96428
  Fold 11/20 bACC=0.96254
  Fold 12/20 bACC=0.95933
  Fold 13/20 bACC=0.96029
  Fold 14/20 bACC=0.96451
  Fold 15/20 bACC=0.96046
  Fold 16/20 bACC=0.96047
  Fold 17/20 bACC=0.95655
  Fold 18/20 bACC=0.96408
  Fold 19/20 bACC=0.95697
  Fold 20/20 bACC=0.95345

cuML RF OOF bACC = 0.96128 +/- 0.00338
Time: 6.8 min


## 12. RealMLP (pytabkit) - Neural Network

In [15]:
print("=" * 65)
print(f"RealMLP | 5-Fold SKF | n_ens=4 | n_epochs=50 (speed-tuned)")
print("=" * 65)


RealMLP_FOLDS = 5
mlp_skf = StratifiedKFold(n_splits=RealMLP_FOLDS, shuffle=True, random_state=SEED)

if not REALMLP_AVAIL:
    print("RealMLP not available. Skipping.")
    oof_mlp  = None
    test_mlp = None
    fold_mlp = []
else:
    oof_mlp  = np.zeros((N_TRAIN, N_CLASSES))
    test_mlp = np.zeros((N_TEST,  N_CLASSES))
    fold_mlp = []
    t0 = time.time()

    for fold, (tri, vai) in enumerate(mlp_skf.split(X_base, y_arr)):
        Xtr_r, Xva_r = X_base.iloc[tri].copy(), X_base.iloc[vai].copy()
        Xte_r        = X_test_b.copy()
        ytr, yva     = y_arr[tri], y_arr[vai]

        Xtr, Xva, Xte, te_c = compute_te_fold(Xtr_r, ytr, Xva_r, Xte_r, CAT_COLS)
        fc = ALL_NUM + te_c

        m = RealMLP_TD_Classifier(
            n_epochs=50,   # was 256 → 62 min/fold; now ~12 min/fold
            n_ens=4,       # was 8 → halved for speed
            use_ls=True,
            device='cuda',
        )
        m.fit(Xtr[fc], ytr)

        oof_p  = m.predict_proba(Xva[fc])
        test_p = m.predict_proba(Xte[fc])

        oof_mlp[vai]  += oof_p
        test_mlp      += test_p / RealMLP_FOLDS

        s = balanced_accuracy_score(yva, np.argmax(oof_p, axis=1))
        fold_mlp.append(s)
        print(f"  Fold {fold+1:>2}/{RealMLP_FOLDS} bACC={s:.5f}")
        del m; gc.collect()

    mlp_cv = np.mean(fold_mlp)
    oof_store['RealMLP']  = oof_mlp
    test_store['RealMLP'] = test_mlp
    cv_scores['RealMLP']  = fold_mlp
    print(f"\nRealMLP OOF bACC = {mlp_cv:.5f} +/- {np.std(fold_mlp):.5f}")
    print(f"Time: {(time.time()-t0)/60:.1f} min")

RealMLP | 5-Fold SKF | n_ens=4 | n_epochs=50 (speed-tuned)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts

  Fold  1/5 bACC=0.96347


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts

  Fold  2/5 bACC=0.96524


...

  Fold  3/5 bACC=0.96230


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts

  Fold  4/5 bACC=0.96325


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts

  Fold  5/5 bACC=0.95869

RealMLP OOF bACC = 0.96259 +/- 0.00217
Time: 60.0 min


## 13. Individual Model CV Summary

In [16]:
print("=" * 65)
print("  INDIVIDUAL MODEL CV SUMMARY")
print("=" * 65)
print(f"  {'Model':<15} {'OOF bACC':>10} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 65)
for mn, folds in sorted(cv_scores.items(), key=lambda x: -np.mean(x[1])):
    a = np.array(folds)
    print(f"  {mn:<15} {a.mean():>10.5f} {a.std():>8.5f} {a.min():>8.5f} {a.max():>8.5f}")
print("=" * 65)

np.save('/kaggle/working/oof_store.npy',  oof_store,  allow_pickle=True)
np.save('/kaggle/working/test_store.npy', test_store, allow_pickle=True)

  INDIVIDUAL MODEL CV SUMMARY
  Model             OOF bACC      Std      Min      Max
-----------------------------------------------------------------
  XGBoost            0.97257  0.00209  0.96789  0.97666
  LightGBM           0.96846  0.00184  0.96501  0.97026
  CatBoost           0.96766  0.00262  0.96084  0.97146
  RealMLP            0.96259  0.00217  0.95869  0.96524
  YDF                0.96136  0.00218  0.95771  0.96370
  cuML_RF            0.96128  0.00338  0.95345  0.96664


## 14. Hill Climbing Ensemble

In [17]:
def hill_climb(oofs_dict, y_true, n_iter=3000, seed=42):
    np.random.seed(seed)
    names  = list(oofs_dict.keys())
    oofs   = [oofs_dict[n] for n in names]
    n      = len(oofs)
    w      = np.ones(n) / n
    blend  = sum(wi * o for wi, o in zip(w, oofs))
    best   = balanced_accuracy_score(y_true, np.argmax(blend, axis=1))
    print(f"Start (equal weights): {best:.5f}")
    for _ in range(n_iter):
        idx   = np.random.randint(0, n)
        delta = np.random.uniform(-0.05, 0.05)
        nw    = w.copy(); nw[idx] += delta
        nw    = np.clip(nw, 0, None)
        if nw.sum() == 0:
            continue
        nw   /= nw.sum()
        blend = sum(wi * o for wi, o in zip(nw, oofs))
        sc    = balanced_accuracy_score(y_true, np.argmax(blend, axis=1))
        if sc > best:
            best = sc; w = nw
    return dict(zip(names, w)), best

print("Running Hill Climbing...")
hc_weights, hc_cv = hill_climb(oof_store, y_arr, n_iter=3000)

print(f"\nHill Climbing ensemble bACC = {hc_cv:.5f}")
print("\nOptimal weights:")
for mn, wt in sorted(hc_weights.items(), key=lambda x: -x[1]):
    print(f"  {mn:<15}: {wt:.4f}")

test_hc = sum(hc_weights.get(m, 0) * test_store[m] for m in oof_store)
preds_hc = [INV_MAP[p] for p in np.argmax(test_hc, axis=1)]

Running Hill Climbing...
Start (equal weights): 0.96438

Hill Climbing ensemble bACC = 0.97257

Optimal weights:
  XGBoost        : 1.0000
  LightGBM       : 0.0000
  CatBoost       : 0.0000
  YDF            : 0.0000
  cuML_RF        : 0.0000
  RealMLP        : 0.0000


## 15. Pseudo Labels (TRES=0.999)

In [18]:
max_p  = test_hc.max(axis=1)
mask   = max_p >= TRES
n_ps   = mask.sum()
print(f"Pseudo label threshold: {TRES}")
print(f"High-confidence test samples: {n_ps:,} / {N_TEST:,} ({n_ps/N_TEST:.1%})")

if n_ps > 0:
    pseudo_y = np.argmax(test_hc[mask], axis=1)
    print(f"  Low={np.sum(pseudo_y==0):,}  Medium={np.sum(pseudo_y==1):,}  High={np.sum(pseudo_y==2):,}")
    PSEUDO_DONE = True
else:
    print("  No pseudo labels at this threshold - threshold is conservative by design")
    PSEUDO_DONE = False

Pseudo label threshold: 0.999
High-confidence test samples: 76,146 / 270,000 (28.2%)
  Low=64,488  Medium=7,978  High=3,680


## 16. Submission Generation

In [19]:
assert not any(np.isnan(v).any() for v in test_store.values()), "NaN in test preds!"
print("Sanity checks passed!")

from scipy.stats import rankdata

def rank_blend_mc(arrays, weights):
    n = len(arrays[0])
    result = np.zeros_like(arrays[0])
    for arr, w in zip(arrays, weights):
        for cls in range(arr.shape[1]):
            result[:, cls] += w * (rankdata(arr[:, cls]) / n)
    return result

src_map = {l: i for i, l in enumerate(INTERNAL_ORDER)}

# ── When HC collapses to single model, just USE that model directly ───────
hc_collapsed = (max(hc_weights.values()) >= 0.99)
best_model   = max(hc_weights, key=hc_weights.get)

if hc_collapsed:
    print(f"HC collapsed to {best_model} — using it directly for all submissions")
    oof_hc   = oof_store[best_model].copy()
    test_hc  = test_store[best_model].copy()
    hc_cv    = np.mean(cv_scores[best_model])
else:
    hc_models = list(hc_weights.keys())
    hc_w      = np.array([hc_weights[m] for m in hc_models])
    test_hc   = rank_blend_mc([test_store[m] for m in hc_models], hc_w)
    oof_hc    = sum(hc_weights[m] * oof_store[m] for m in hc_weights)
    hc_cv     = balanced_accuracy_score(y_arr, np.argmax(oof_hc, axis=1))

print(f"\nHC OOF bACC: {hc_cv:.5f}")

# ── Bias tuning on HC blend (which is XGB alone if collapsed) ────────────
print("\nBias tuning HC blend...")
hc_bias, hc_bias_score, hc_oof_pub, hc_y_pub = tune_bias(oof_hc, y_arr)

test_hc_pub = test_hc[:, [src_map[l] for l in PUBLIC_ORDER]]
test_labels  = public_preds_with_bias(test_hc_pub, hc_bias)
decode = np.array(PUBLIC_ORDER)

pred_counts = {cls: int(np.sum(test_labels == i)) for i, cls in enumerate(PUBLIC_ORDER)}
print(f"Test dist (biased): {pred_counts}")

high_pct = pred_counts.get('High', 0) / N_TEST
if high_pct > 0.10:
    print(f"WARNING: High={high_pct:.1%} too aggressive → raw argmax")
    hc_bias = np.zeros(N_CLASSES)
    hc_bias_score = balanced_accuracy_score(hc_y_pub, public_preds_with_bias(hc_oof_pub, hc_bias))
    test_labels = public_preds_with_bias(test_hc_pub, hc_bias)
    pred_counts = {cls: int(np.sum(test_labels == i)) for i, cls in enumerate(PUBLIC_ORDER)}

# ── V1: HC + Bias ─────────────────────────────────────────────────────────
sub_v1 = sample.copy()
sub_v1['Irrigation_Need'] = decode[test_labels]
sub_v1.to_csv('submission_v1_hc_bias.csv', index=False)
cv_v1 = hc_bias_score
print(f"\nV1 (HC+Bias): CV={cv_v1:.5f} dist={pred_counts}")

# ── V2: HC raw (no bias) ─────────────────────────────────────────────────
raw_labels = np.array([I2P[v] for v in np.argmax(test_hc, axis=1)])
sub_v2 = sample.copy()
sub_v2['Irrigation_Need'] = decode[raw_labels]
sub_v2.to_csv('submission_v2_hc_raw.csv', index=False)
cv_v2 = hc_cv
print(f"V2 (HC Raw):  CV={cv_v2:.5f} dist={sub_v2['Irrigation_Need'].value_counts().to_dict()}")

# ── V3: XGB + Bias ────────────────────────────────────────────────────────
xgb_bias, xgb_bias_score, xgb_oof_pub, xgb_y_pub = tune_bias(oof_store['XGBoost'], y_arr)
xgb_test_pub = test_store['XGBoost'][:, [src_map[l] for l in PUBLIC_ORDER]]
xgb_labels   = public_preds_with_bias(xgb_test_pub, xgb_bias)

xgb_high_pct = float(np.sum(xgb_labels == 0)) / N_TEST
if xgb_high_pct > 0.10:
    print(f"V3 XGB bias High={xgb_high_pct:.1%} → raw argmax")
    xgb_labels = np.array([I2P[v] for v in np.argmax(test_store['XGBoost'], axis=1)])
    xgb_bias_score = balanced_accuracy_score(y_arr, np.argmax(oof_store['XGBoost'], axis=1))

sub_v3 = sample.copy()
sub_v3['Irrigation_Need'] = decode[xgb_labels]
sub_v3.to_csv('submission_v3_xgb_bias.csv', index=False)
cv_v3 = xgb_bias_score
print(f"V3 (XGB+Bias):CV={cv_v3:.5f} dist={sub_v3['Irrigation_Need'].value_counts().to_dict()}")

print(f"\n>>> SUBMIT ORDER: V3 first (best CV), then V1, then V2")

Sanity checks passed!
HC collapsed to XGBoost — using it directly for all submissions

HC OOF bACC: 0.97257

Bias tuning HC blend...
  Before bias: 0.97257
  After  bias: 0.97341 | bias=[ 0.4 -0.5  0. ]
Test dist (biased): {'High': 10721, 'Low': 159856, 'Medium': 99423}

V1 (HC+Bias): CV=0.97341 dist={'High': 10721, 'Low': 159856, 'Medium': 99423}
V2 (HC Raw):  CV=0.97257 dist={'Low': 159955, 'Medium': 100159, 'High': 9886}
  Before bias: 0.97257
  After  bias: 0.97341 | bias=[ 0.4 -0.5  0. ]
V3 (XGB+Bias):CV=0.97341 dist={'Low': 159856, 'Medium': 99423, 'High': 10721}

>>> SUBMIT ORDER: V3 first (best CV), then V1, then V2


In [20]:
# ── OPTUNA CLASS WEIGHT OPTIMIZATION ──────────────────────────────────────
# From discussion: multiplying proba by class weights before argmax
# boosts CV by ~0.002 guaranteed
print("Running Optuna class weight optimization on XGB OOF...")

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from scipy.stats import rankdata

# Use XGB OOF (best single model)
xgb_oof_internal = oof_store['XGBoost'].copy()   # shape (N_TRAIN, 3) — internal order
# Map to public order: internal=[Low=0,Med=1,High=2], public=[High=0,Low=1,Med=2]
src_map = {l: i for i, l in enumerate(INTERNAL_ORDER)}
pub_idx  = [src_map[l] for l in PUBLIC_ORDER]   # [2, 0, 1]
xgb_oof_pub = xgb_oof_internal[:, pub_idx]       # now columns = High, Low, Med

def optuna_objective(trial):
    cw0 = trial.suggest_float('cw0', 0.5, 3.0)   # High
    cw1 = trial.suggest_float('cw1', 0.5, 3.0)   # Low
    cw2 = trial.suggest_float('cw2', 0.5, 3.0)   # Medium
    cw  = np.array([cw0, cw1, cw2])
    adj = xgb_oof_pub * cw
    adj = adj / adj.sum(axis=1, keepdims=True)
    # y_pub: convert internal y_arr to public labels
    y_pub_arr = np.array([I2P[v] for v in y_arr])
    return balanced_accuracy_score(y_pub_arr, np.argmax(adj, axis=1))

study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(optuna_objective, n_trials=300, show_progress_bar=False)

best_cw  = np.array([study.best_params['cw0'],
                     study.best_params['cw1'],
                     study.best_params['cw2']])
best_cv_optuna = study.best_value
print(f"Optuna best bACC = {best_cv_optuna:.5f}")
print(f"Best weights: High={best_cw[0]:.4f} Low={best_cw[1]:.4f} Med={best_cw[2]:.4f}")

# Apply to test
xgb_test_pub = test_store['XGBoost'][:, pub_idx]
adj_test = xgb_test_pub * best_cw
adj_test = adj_test / adj_test.sum(axis=1, keepdims=True)
optuna_labels = np.argmax(adj_test, axis=1)
decode = np.array(PUBLIC_ORDER)

sub_optuna = sample.copy()
sub_optuna['Irrigation_Need'] = decode[optuna_labels]
sub_optuna.to_csv('submission_v4_optuna_cw.csv', index=False)
dist_v4 = sub_optuna['Irrigation_Need'].value_counts().to_dict()
print(f"V4 (XGB+Optuna CW): bACC={best_cv_optuna:.5f}  dist={dist_v4}")


Running Optuna class weight optimization on XGB OOF...
Optuna best bACC = 0.97362
Best weights: High=2.1383 Low=1.2475 Med=1.2406
V4 (XGB+Optuna CW): bACC=0.97362  dist={'Low': 159955, 'Medium': 98951, 'High': 11094}


## 17. CV-LB Tracking

In [21]:
# ── CV-LB Tracking Table ─────────────────────────────────────────────────
tracking = pd.DataFrame([
    {'Version':'V1', 'CV': round(cv_v1,5), 'Public_LB': None,
     'Notes': f'HC({best_model})+Bias SEED=42'},
    {'Version':'V2', 'CV': round(cv_v2,5), 'Public_LB': None,
     'Notes': f'HC({best_model}) Raw argmax'},
    {'Version':'V3', 'CV': round(cv_v3,5), 'Public_LB': 0.97242,
     'Notes': 'XGB alone + Bias  ← BEST SO FAR'},
    {'Version':'V4', 'CV': round(best_cv_optuna,5), 'Public_LB': None,
     'Notes': 'XGB + Optuna class weights ← NEW'},
])
print("=" * 65)
print("CV-LB TRACKING")
print("=" * 65)
print(tracking.to_string(index=False))
print(f"\nXGB CV: {np.mean(cv_scores['XGBoost']):.5f} | Best LB so far: 0.97242")
print(f"CV-LB gap (XGB+Bias): {cv_v3 - 0.97242:.5f}")
print(f"\n>>> SUBMIT: V4 first (Optuna CW), then V3, then V1")


CV-LB TRACKING
Version      CV  Public_LB                            Notes
     V1 0.97341        NaN         HC(XGBoost)+Bias SEED=42
     V2 0.97257        NaN           HC(XGBoost) Raw argmax
     V3 0.97341    0.97242  XGB alone + Bias  ← BEST SO FAR
     V4 0.97362        NaN XGB + Optuna class weights ← NEW

XGB CV: 0.97257 | Best LB so far: 0.97242
CV-LB gap (XGB+Bias): 0.00099

>>> SUBMIT: V4 first (Optuna CW), then V3, then V1
